# triangle-splatting :: Custom data :: Celebrity_Gym

-----
- Conda env : [waikiki_statue](README.md#setup-a-conda-environment)
-----

### Check system

In [1]:
!nvidia-smi

Sat Oct  4 05:03:35 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:01:00.0  On |                  N/A |
| 24%   43C    P8             32W /  250W |     539MiB /  11264MiB |     23%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Download a video

In [2]:
import os
from pathlib import Path

Path("./temp_data").mkdir(exist_ok=True, parents=True)

In [3]:
import gdown
VIDEO_NAME = "Celebrity_Gym"

FPS = 10
RES = 4
id = "1jx1jtZQnxe1lmzzKQTBKuffmic4Did62"
vid_path = f"./temp_data/{VIDEO_NAME}.mov"

gdown.download(id=id, output = vid_path)

Downloading...
From: https://drive.google.com/uc?id=1jx1jtZQnxe1lmzzKQTBKuffmic4Did62
To: /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/triangle_splatting/temp_data/Celebrity_Gym.mov
100%|██████████| 16.9M/16.9M [00:01<00:00, 12.1MB/s]


'./temp_data/Celebrity_Gym.mov'

### Extract images from the video

In [4]:
DATASET_DIR_PATH = f"./temp_data/{VIDEO_NAME}"
IMAGES_DIR_PATH = os.path.join(DATASET_DIR_PATH, "images")
DATABASE_PATH = os.path.join(DATASET_DIR_PATH, "database.db")

Path(IMAGES_DIR_PATH).mkdir(exist_ok=True, parents=True)


!ffmpeg -i $vid_path -vf fps=$FPS $IMAGES_DIR_PATH/frame_%04d.jpg

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

### Colmap :: Feature Extraction

In [5]:
# Colmap Feature Extraction
!colmap feature_extractor \
    --database_path $DATABASE_PATH \
    --image_path $IMAGES_DIR_PATH  --ImageReader.camera_model PINHOLE


Feature extraction

Processed file [1/153]
  Name:            frame_0001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - PINHOLE
  Focal Length:    2304.00px
  Features:        793
Processed file [2/153]
  Name:            frame_0002.jpg
  Dimensions:      1920 x 1080
  Camera:          #2 - PINHOLE
  Focal Length:    2304.00px
  Features:        805
Processed file [3/153]
  Name:            frame_0003.jpg
  Dimensions:      1920 x 1080
  Camera:          #3 - PINHOLE
  Focal Length:    2304.00px
  Features:        673
Processed file [4/153]
  Name:            frame_0004.jpg
  Dimensions:      1920 x 1080
  Camera:          #4 - PINHOLE
  Focal Length:    2304.00px
  Features:        718
Processed file [5/153]
  Name:            frame_0005.jpg
  Dimensions:      1920 x 1080
  Camera:          #5 - PINHOLE
  Focal Length:    2304.00px
  Features:        738
Processed file [6/153]
  Name:            frame_0006.jpg
  Dimensions:      1920 x 1080
  Camera:          #6 - PINHOLE

### Colmap :: Feature Matching

In [6]:
# Feature Matching
!colmap sequential_matcher \
    --database_path $DATABASE_PATH


Sequential feature matching

Matching image [1/153] in 0.298s
Matching image [2/153] in 0.146s
Matching image [3/153] in 0.110s
Matching image [4/153] in 0.239s
Matching image [5/153] in 0.128s
Matching image [6/153] in 0.289s
Matching image [7/153] in 0.192s
Matching image [8/153] in 0.113s
Matching image [9/153] in 0.168s
Matching image [10/153] in 0.145s
Matching image [11/153] in 0.200s
Matching image [12/153] in 0.211s
Matching image [13/153] in 0.201s
Matching image [14/153] in 0.185s
Matching image [15/153] in 0.102s
Matching image [16/153] in 0.174s
Matching image [17/153] in 0.141s
Matching image [18/153] in 0.110s
Matching image [19/153] in 0.149s
Matching image [20/153] in 0.146s
Matching image [21/153] in 0.154s
Matching image [22/153] in 0.181s
Matching image [23/153] in 0.276s
Matching image [24/153] in 0.225s
Matching image [25/153] in 0.188s
Matching image [26/153] in 0.260s
Matching image [27/153] in 0.195s
Matching image [28/153] in 0.320s
Matching image [29/153] in 

### Colmap :: Sparse Reconstruction (Mapper)

In [7]:
# Sparse Reconstruction (Mapper)
SPARSE_DIR = os.path.join(DATASET_DIR_PATH, "sparse")
Path(SPARSE_DIR).mkdir(exist_ok=True, parents=True)

!colmap mapper \
    --database_path $DATABASE_PATH \
    --image_path $IMAGES_DIR_PATH \
    --output_path $SPARSE_DIR


Loading database

Loading cameras... 153 in 0.000s
Loading matches... 1636 in 0.014s
Loading images... 153 in 0.015s (connected 153)
Building correspondence graph... in 0.054s (ignored 0)

Elapsed time: 0.001 [minutes]


Finding good initial image pair

  => No good initial image pair found.
  => Relaxing the initialization constraints.

Finding good initial image pair


Initializing with image pair #99 and #115


Global bundle adjustment

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  2.238437e+02    0.00e+00    2.01e+04   0.00e+00   0.00e+00  1.00e+04        0    1.13e-04    6.76e-04
   1  3.736515e+02   -1.50e+02    2.01e+04   2.09e+02  -2.30e+00  5.00e+03        1    1.32e-04    8.21e-04
   2  2.337027e+02   -9.86e+00    2.01e+04   1.54e+02  -1.71e-01  1.25e+03        1    7.20e-05    9.00e-04
   3  1.880897e+02    3.58e+01    4.37e+03   6.42e+01   9.52e-01  3.75e+03        1    1.42e-04    1.05e-03
   4  1.813753e

### Triangle-Splatting :: Training (Indoor mode)

In [8]:
DATASET_DIR_PATH
OUTPUR_DIR_PATH = f"./temp_result/{VIDEO_NAME}"
print(DATASET_DIR_PATH)
print(OUTPUR_DIR_PATH)

!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/train.py -s $DATASET_DIR_PATH -m $OUTPUR_DIR_PATH -r $RES --eval

./temp_data/Celebrity_Gym
./temp_result/Celebrity_Gym
Optimizing ./temp_result/Celebrity_Gym
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth
Output folder: ./temp

### Triangle-Splatting :: Rendering (Indoor mode)

In [9]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/render.py -m $OUTPUR_DIR_PATH

Looking for config file in ./temp_result/Celebrity_Gym/cfg_args
Config file found: ./temp_result/Celebrity_Gym/cfg_args
Rendering ./temp_result/Celebrity_Gym
Loading trained model at iteration 30000 [04/10 05:30:02]
Reading camera 1/153----- PINHOLE [04/10 05:30:02]
Reading camera 2/153----- PINHOLE [04/10 05:30:02]
Reading camera 3/153----- PINHOLE [04/10 05:30:02]
Reading camera 4/153----- PINHOLE [04/10 05:30:02]
Reading camera 5/153----- PINHOLE [04/10 05:30:02]
Reading camera 6/153----- PINHOLE [04/10 05:30:02]
Reading camera 7/153----- PINHOLE [04/10 05:30:02]
Reading camera 8/153----- PINHOLE [04/10 05:30:02]
Reading camera 9/153----- PINHOLE [04/10 05:30:02]
Reading camera 10/153----- PINHOLE [04/10 05:30:02]
Reading camera 11/153----- PINHOLE [04/10 05:30:02]
Reading camera 12/153----- PINHOLE [04/10 05:30:02]
Reading camera 13/153----- PINHOLE [04/10 05:30:02]
Reading camera 14/153----- PINHOLE [04/10 05:30:02]
Reading camera 15/153----- PINHOLE [04/10 05:30:02]
Reading camer

### Triangle-Splatting :: Create a video (Indoor mode)

In [10]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/create_video.py -m $OUTPUR_DIR_PATH

Looking for config file in ./temp_result/Celebrity_Gym/cfg_args
Config file found: ./temp_result/Celebrity_Gym/cfg_args
Creating video for ./temp_result/Celebrity_Gym
Loading trained model at iteration 30000
Reading camera 1/153----- PINHOLE
Reading camera 2/153----- PINHOLE
Reading camera 3/153----- PINHOLE
Reading camera 4/153----- PINHOLE
Reading camera 5/153----- PINHOLE
Reading camera 6/153----- PINHOLE
Reading camera 7/153----- PINHOLE
Reading camera 8/153----- PINHOLE
Reading camera 9/153----- PINHOLE
Reading camera 10/153----- PINHOLE
Reading camera 11/153----- PINHOLE
Reading camera 12/153----- PINHOLE
Reading camera 13/153----- PINHOLE
Reading camera 14/153----- PINHOLE
Reading camera 15/153----- PINHOLE
Reading camera 16/153----- PINHOLE
Reading camera 17/153----- PINHOLE
Reading camera 18/153----- PINHOLE
Reading camera 19/153----- PINHOLE
Reading camera 20/153----- PINHOLE
Reading camera 21/153----- PINHOLE
Reading camera 22/153----- PINHOLE
Reading camera 23/153----- PINH

### Triangle-Splatting :: Training (Outdoor mode)

In [11]:
DATASET_DIR_PATH
OUTDOOR_OUTPUR_DIR_PATH = f"./temp_result/{VIDEO_NAME}_outdoor"
print(DATASET_DIR_PATH)
print(OUTDOOR_OUTPUR_DIR_PATH)

!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/train.py -s $DATASET_DIR_PATH -m $OUTDOOR_OUTPUR_DIR_PATH -r $RES --eval  --outdoor 

./temp_data/Celebrity_Gym
./temp_result/Celebrity_Gym_outdoor
Optimizing ./temp_result/Celebrity_Gym_outdoor
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth
Outpu

### Triangle-Splatting :: Rendering (Outdoor mode)

In [12]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/render.py -m $OUTDOOR_OUTPUR_DIR_PATH

Looking for config file in ./temp_result/Celebrity_Gym_outdoor/cfg_args
Config file found: ./temp_result/Celebrity_Gym_outdoor/cfg_args
Rendering ./temp_result/Celebrity_Gym_outdoor
Loading trained model at iteration 30000 [04/10 05:50:54]
Reading camera 1/153----- PINHOLE [04/10 05:50:54]
Reading camera 2/153----- PINHOLE [04/10 05:50:54]
Reading camera 3/153----- PINHOLE [04/10 05:50:54]
Reading camera 4/153----- PINHOLE [04/10 05:50:54]
Reading camera 5/153----- PINHOLE [04/10 05:50:54]
Reading camera 6/153----- PINHOLE [04/10 05:50:54]
Reading camera 7/153----- PINHOLE [04/10 05:50:54]
Reading camera 8/153----- PINHOLE [04/10 05:50:54]
Reading camera 9/153----- PINHOLE [04/10 05:50:54]
Reading camera 10/153----- PINHOLE [04/10 05:50:54]
Reading camera 11/153----- PINHOLE [04/10 05:50:54]
Reading camera 12/153----- PINHOLE [04/10 05:50:54]
Reading camera 13/153----- PINHOLE [04/10 05:50:54]
Reading camera 14/153----- PINHOLE [04/10 05:50:54]
Reading camera 15/153----- PINHOLE [04/10

### Triangle-Splatting :: Create a video (Outdoor mode)

In [13]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/create_video.py -m $OUTDOOR_OUTPUR_DIR_PATH

Looking for config file in ./temp_result/Celebrity_Gym_outdoor/cfg_args
Config file found: ./temp_result/Celebrity_Gym_outdoor/cfg_args
Creating video for ./temp_result/Celebrity_Gym_outdoor
Loading trained model at iteration 30000
Reading camera 1/153----- PINHOLE
Reading camera 2/153----- PINHOLE
Reading camera 3/153----- PINHOLE
Reading camera 4/153----- PINHOLE
Reading camera 5/153----- PINHOLE
Reading camera 6/153----- PINHOLE
Reading camera 7/153----- PINHOLE
Reading camera 8/153----- PINHOLE
Reading camera 9/153----- PINHOLE
Reading camera 10/153----- PINHOLE
Reading camera 11/153----- PINHOLE
Reading camera 12/153----- PINHOLE
Reading camera 13/153----- PINHOLE
Reading camera 14/153----- PINHOLE
Reading camera 15/153----- PINHOLE
Reading camera 16/153----- PINHOLE
Reading camera 17/153----- PINHOLE
Reading camera 18/153----- PINHOLE
Reading camera 19/153----- PINHOLE
Reading camera 20/153----- PINHOLE
Reading camera 21/153----- PINHOLE
Reading camera 22/153----- PINHOLE
Reading